# 01 · Dummy's first call – models and the Responses API

In the garage, Tony's first helper was a clumsy robot arm called **Dummy**. It could do exactly one thing at a time. That is where we start too: **one model, one question, one answer**.

You will see:
1. the **Responses API** (the API agents are built on) and the older **Chat Completions** API
2. **streaming**, **instructions** and **structured output**
3. what a response object really contains (tokens, output items)
4. **embeddings**

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

oai = openai_client()      # OpenAI client bound to the Foundry project
MODEL = model("chat")
print("model deployment:", MODEL)

## 1 · The Responses API
`model` is the **deployment name**. `input` can be a string or a list of messages.

In [ ]:
r = oai.responses.create(model=MODEL, input="In one sentence: what is Dummy in Iron Man?")
show_output(r)

### Instructions (the system prompt) and limits
`instructions` steers behaviour for this call only. `max_output_tokens` caps cost.

In [ ]:
r = oai.responses.create(
    model=MODEL,
    instructions="You are Dummy, a clumsy robot arm. Answer in at most 12 words and end with a beep.",
    input="Did you spill the smoothie again?",
    max_output_tokens=200,
)
show_output(r)

## 2 · Chat Completions (the classic shape)
The same deployment also speaks Chat Completions. New work should use Responses: it is stateful, supports tools and agents. Chat Completions is still handy for existing code.

In [ ]:
c = oai.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Name the three MCU AIs that came before Vision. Short list."}],
)
print(c.choices[0].message.content)

## 3 · Streaming
Events arrive as the model writes. `response.output_text.delta` carries the text pieces.

In [ ]:
stream = oai.responses.create(model=MODEL, input="Write a 3-line status report of a workshop fire drill.", stream=True)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        print("\n\n[done] tokens:", event.response.usage.total_tokens)

## 4 · Structured output
Ask for JSON that matches a schema. `strict` makes the model follow it exactly, so your code can parse the answer without guessing.

In [ ]:
import json
schema = {
    "type": "object",
    "properties": {
        "suit": {"type": "string"},
        "mark": {"type": "integer"},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["suit", "mark", "weaknesses"],
    "additionalProperties": False,
}
r = oai.responses.create(
    model=MODEL,
    input="Describe the Mark 3 armor.",
    text={"format": {"type": "json_schema", "name": "suit_card", "schema": schema, "strict": True}},
)
card = json.loads(r.output_text)
print(card)

## 5 · What is inside a response?
A response is more than text: `output` is a list of **items** (messages, tool calls, reasoning), `usage` counts tokens (this is what you pay for).

In [ ]:
print("status :", r.status)
print("usage  :", r.usage.input_tokens, "in /", r.usage.output_tokens, "out /", r.usage.total_tokens, "total")
for item in r.output:
    print("item   :", item.type)

## 6 · Embeddings
Embeddings turn text into vectors for search and retrieval. This uses the embedding deployment, not the chat model.

In [ ]:
e = oai.embeddings.create(model=model("embedding"), input=["arc reactor", "repulsor", "smoothie"])
vecs = [d.embedding for d in e.data]
print(len(vecs), "vectors of", len(vecs[0]), "dimensions")

import math
cos = lambda a, b: sum(x*y for x, y in zip(a, b)) / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))
print("arc reactor ~ repulsor :", round(cos(vecs[0], vecs[1]), 3))
print("arc reactor ~ smoothie :", round(cos(vecs[0], vecs[2]), 3))

## Recap
| Need | Use |
|---|---|
| One-off answer, new code | `responses.create(model=..., input=...)` |
| Existing Chat Completions code | `chat.completions.create(...)` |
| Parseable output | `text={"format": {"type": "json_schema", ...}}` |
| Search / similarity | `embeddings.create(...)` |

Dummy has no memory: every call starts from zero. > 💡 Next: [02 · Conversations](02_conversations.ipynb)